In [1]:
# Install dependencies
!pip install plotly ucimlrepo


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px

from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

# Load Dataset
wine = fetch_ucirepo(id=109)

X = wine.data.features
y = wine.data.targets.squeeze()

# Standardize Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

# Original Classification
clf = LogisticRegression(max_iter=500)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

orig_acc = accuracy_score(y_test, y_pred)
orig_f1 = f1_score(y_test, y_pred, average='weighted')

# LDA (1 Component)
lda1 = LinearDiscriminantAnalysis(n_components=1)
X_train_lda1 = lda1.fit_transform(X_train, y_train)
X_test_lda1 = lda1.transform(X_test)

clf1 = LogisticRegression(max_iter=500)
clf1.fit(X_train_lda1, y_train)
y_pred1 = clf1.predict(X_test_lda1)

lda1_acc = accuracy_score(y_test, y_pred1)
lda1_f1 = f1_score(y_test, y_pred1, average='weighted')

# LDA (2 Components)

lda2 = LinearDiscriminantAnalysis(n_components=2)
X_train_lda2 = lda2.fit_transform(X_train, y_train)
X_test_lda2 = lda2.transform(X_test)

clf2 = LogisticRegression(max_iter=500)
clf2.fit(X_train_lda2, y_train)
y_pred2 = clf2.predict(X_test_lda2)

lda2_acc = accuracy_score(y_test, y_pred2)
lda2_f1 = f1_score(y_test, y_pred2, average='weighted')


# Save Projection

X_lda_full = lda2.fit_transform(X_scaled, y)

projection_df = pd.DataFrame(X_lda_full, columns=["LD1", "LD2"])
projection_df["Class"] = y.values
projection_df["SampleId"] = np.arange(len(projection_df))

projection_df = projection_df[["SampleId", "LD1", "LD2", "Class"]]
projection_df.to_csv("lda_projection.csv", index=False)


# Plotly Visualization

fig = px.scatter(
    projection_df,
    x="LD1",
    y="LD2",
    color=projection_df["Class"].astype(str),
    title="LDA Projection (2D)",
    labels={"color": "Class"},
    hover_data=["SampleId"]
)

fig.show()


# PRINT OUTPUTS

print("\n\t===== CLASSIFICATION RESULTS =====")
print(f"Original Data -> Accuracy: {orig_acc:.4f}, F1 Score: {orig_f1:.4f}")
print(f"LDA (1 Component) -> Accuracy: {lda1_acc:.4f}, F1 Score: {lda1_f1:.4f}")
print(f"LDA (2 Components) -> Accuracy: {lda2_acc:.4f}, F1 Score: {lda2_f1:.4f}")

print("\n\t===== FILE GENERATED =====")
print("lda_projection.csv saved successfully.")


# SHORT REPORT

print("\n\t===== SHORT REPORT =====")
print("""
Class Separability:
LDA projects the data by maximizing separation between wine classes using label information.
In 1D projection, some overlap remains because only one discriminant axis is used.
However, in 2D projection, the classes become clearly separable and form distinct clusters,
indicating that LDA effectively captures the class-dependent structure of the dataset.

Classification Performance:
The original standardized dataset achieves very high classification accuracy.
Reducing to 1 LDA component causes a slight drop in performance due to information loss.
However, using 2 components restores performance close to the original dataset,
demonstrating that LDA can reduce dimensionality while preserving most discriminative information.

LDA vs PCA:
LDA is a supervised technique that uses class labels to maximize class separability,
whereas PCA is an unsupervised method that focuses only on maximizing variance.
As a result, PCA may not effectively separate classes, while LDA produces projections
that are more suitable for classification tasks.
""")


	===== CLASSIFICATION RESULTS =====
Original Data -> Accuracy: 0.9815, F1 Score: 0.9815
LDA (1 Component) -> Accuracy: 0.9074, F1 Score: 0.9079
LDA (2 Components) -> Accuracy: 0.9815, F1 Score: 0.9815

	===== FILE GENERATED =====
lda_projection.csv saved successfully.

	===== SHORT REPORT =====

Class Separability:
LDA projects the data by maximizing separation between wine classes using label information.
In 1D projection, some overlap remains because only one discriminant axis is used.
However, in 2D projection, the classes become clearly separable and form distinct clusters,
indicating that LDA effectively captures the class-dependent structure of the dataset.

Classification Performance:
The original standardized dataset achieves very high classification accuracy.
Reducing to 1 LDA component causes a slight drop in performance due to information loss.
However, using 2 components restores performance close to the original dataset,
demonstrating that LDA can reduce dimensionality wh